# **Project Name** - Yes Bank Stock Price Prediction

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

### Dataset Loading

In [ ]:
data_path = "data_YesBank_StockPrices.csv"
df = pd.read_csv(data_path)
df.head()

### Dataset First View

In [ ]:
df.head()


### Dataset Rows & Columns count

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

### Dataset Information

In [ ]:
df.info()


#### Duplicate Values

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

#### Missing Values/Null Values

In [ ]:
print(df.isnull().sum())
print("\nTotal missing values:", df.isnull().sum().sum())

In [ ]:
missing = df.isnull().sum()
plt.figure(figsize=(8,4))
missing.plot(kind='bar')
plt.title('Missing Values by Column')
plt.ylabel('Number of Missing Values')
plt.xticks(rotation=0)
plt.show()

### Dataset understanding
- The dataset contains **185 monthly observations** of Yes Bank stock prices from **Jul-2005 to Nov-2020**.
- It contains five columns: `Date`, `Open`, `High`, `Low`, and `Close`.
- There are **no missing values** and **no duplicate rows**.
- The objective of the ML section is to predict the **next month's closing price** using information available up to the current month.
- Because this is a time-ordered dataset, chronological splitting is used instead of random shuffling to avoid future-information leakage.

The dataset is a monthly historical stock-price series. `Open`, `High`, `Low`, and `Close` are continuous numerical variables, while `Date` is a time identifier. The target is created as the **next month's `Close`**. This makes the ML task a supervised regression problem while respecting the temporal nature of the data.

## ***2. Understanding Your Variables***

In [ ]:
print(df.columns.tolist())

In [ ]:
df.describe(include='all').T

### Variables Description

| Variable | Description | Type |
|---|---|---|
| `Date` | Month and year of the stock-price observation | Date/time |
| `Open` | Opening price for the month | Numerical |
| `High` | Highest price during the month | Numerical |
| `Low` | Lowest price during the month | Numerical |
| `Close` | Closing price for the month | Numerical / prediction source |

The dataset does not contain categorical variables or textual review fields.

### Check Unique Values for each variable.

In [ ]:
for col in df.columns:
    print(f"\n{col}:")
    print("Unique values:", df[col].nunique())
    if col != 'Date':
        print(df[col].describe()[['min','max','mean','50%']].to_dict())

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Convert date and sort chronologically
df['Date'] = pd.to_datetime(df['Date'], format='%b-%y')
df = df.sort_values('Date').reset_index(drop=True)

# Time-derived and market-behaviour features for EDA
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Monthly_Return_%'] = df['Close'].pct_change() * 100
df['Price_Range'] = df['High'] - df['Low']
df['Open_Close_Change'] = df['Close'] - df['Open']
df['Open_Close_Change_%'] = (df['Close'] - df['Open']) / df['Open'] * 100

print(df.head())
print("\nDate range:", df['Date'].min().date(), "to", df['Date'].max().date())

### What all manipulations have you done and insights you found?

### Data-wrangling decisions
1. Converted `Date` from the source string format (`Mon-YY`) into a datetime column.
2. Sorted observations chronologically.
3. Verified missing values and duplicates.
4. Created time-derived variables such as year and month for analysis.
5. Created market-behaviour features such as monthly range, open-to-close change, and percentage return.
6. For ML, created lagged and rolling features using only current and historical observations.
7. Created `Target_Next_Close` with `Close.shift(-1)` and removed the final row because it has no known next-month target.
8. Used chronological train/test splitting to prevent future observations from entering model training.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(df['Date'], df['Close'], linewidth=2)
plt.title('Yes Bank Monthly Closing Price Over Time')
plt.xlabel('Date'); plt.ylabel('Close Price')
plt.show()

##### 1. Why did you pick the specific chart?

A line chart is appropriate because the observations are ordered in time. It reveals long-term trends, peaks, falls and regime changes in the closing price.

##### 2. What is/are the insight(s) found from the chart?

The closing price changes substantially over the sample period, with pronounced rises and declines rather than a stable level. The later period includes a sharp fall and partial recovery.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The trend is useful for understanding historical risk and identifying periods requiring closer investigation. Large drawdowns can indicate significant downside risk, so they should not be interpreted as positive business growth.

#### Chart - 2

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(df['Date'], df['Open'], label='Open')
plt.plot(df['Date'], df['High'], label='High')
plt.plot(df['Date'], df['Low'], label='Low')
plt.plot(df['Date'], df['Close'], label='Close')
plt.title('Monthly OHLC Price Movement')
plt.xlabel('Date'); plt.ylabel('Price')
plt.legend()
plt.show()

##### 1. Why did you pick the specific chart?

Multiple line series allow the four OHLC measures to be compared on the same time axis and show how monthly trading ranges evolved.

##### 2. What is/are the insight(s) found from the chart?

Open, High, Low and Close generally move together, while the distance between High and Low expands during more volatile periods.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The chart can help risk teams understand periods of unusually wide price ranges. Large ranges imply greater uncertainty and potential risk rather than guaranteed business improvement.

#### Chart - 3

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(df['Date'], df['Monthly_Return_%'])
plt.axhline(0, linestyle='--')
plt.title('Monthly Percentage Return')
plt.xlabel('Date'); plt.ylabel('Return (%)')
plt.show()

##### 1. Why did you pick the specific chart?

Monthly returns are more useful than raw price levels for examining percentage movement over time and highlighting positive and negative periods.

##### 2. What is/are the insight(s) found from the chart?

Returns fluctuate around zero with several pronounced positive and negative observations. The variation indicates that monthly price changes are not constant.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Return volatility is important for risk management. Large negative returns can create substantial losses, while positive returns indicate historical appreciation but do not guarantee future performance.

#### Chart - 4

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(df['Date'], df['Price_Range'])
plt.title('Monthly High-Low Price Range')
plt.xlabel('Date'); plt.ylabel('High - Low')
plt.show()

##### 1. Why did you pick the specific chart?

The High-Low range directly measures the monthly price spread and is a simple proxy for price variability.

##### 2. What is/are the insight(s) found from the chart?

The range is not constant. Some periods show much wider spreads, indicating stronger within-month price movement.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Identifying high-range periods can improve risk monitoring and scenario analysis. Wide ranges represent uncertainty and can increase the potential size of both gains and losses.

#### Chart - 5

In [ ]:
rolling_mean = df['Close'].rolling(6).mean()
plt.figure(figsize=(14,5))
plt.plot(df['Date'], df['Close'], alpha=0.5, label='Close')
plt.plot(df['Date'], rolling_mean, linewidth=2, label='6-Month Rolling Mean')
plt.title('Closing Price with 6-Month Rolling Mean')
plt.xlabel('Date'); plt.ylabel('Price')
plt.legend()
plt.show()

##### 1. Why did you pick the specific chart?

A rolling mean smooths short-term fluctuations and makes longer-term movement easier to see without discarding the original observations.

##### 2. What is/are the insight(s) found from the chart?

The rolling average follows broad changes in the closing-price level while being less sensitive to individual monthly spikes or drops.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

A smoothed trend can support historical planning and risk review. However, it is descriptive and should not be treated as a trading signal by itself.

#### Chart - 6

In [ ]:
rolling_vol = df['Monthly_Return_%'].rolling(6).std()
plt.figure(figsize=(14,5))
plt.plot(df['Date'], rolling_vol)
plt.title('6-Month Rolling Volatility of Monthly Returns')
plt.xlabel('Date'); plt.ylabel('Standard Deviation of Return (%)')
plt.show()

##### 1. Why did you pick the specific chart?

Rolling standard deviation of returns provides a local measure of volatility and shows when market movements became more or less variable.

##### 2. What is/are the insight(s) found from the chart?

Volatility changes across time, with clusters of higher and lower variation rather than a constant level.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Volatility clustering matters for risk controls because periods of elevated uncertainty can require more conservative decisions.

#### Chart - 7

In [ ]:
plt.figure(figsize=(10,5))
plt.hist(df['Close'], bins=25)
plt.title('Distribution of Monthly Closing Prices')
plt.xlabel('Close Price'); plt.ylabel('Frequency')
plt.show()

##### 1. Why did you pick the specific chart?

A histogram summarizes the distribution of closing prices and helps identify concentration, spread and skewness.

##### 2. What is/are the insight(s) found from the chart?

Closing prices are spread across a wide range rather than concentrated around one narrow value. This reflects the changing price regime across the long sample period.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Understanding the distribution helps assess model error scales and the presence of extreme price levels. Extreme observations should not automatically be removed because they may represent real market conditions.

#### Chart - 8

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df[['Open','High','Low','Close']])
plt.title('Boxplot of OHLC Prices')
plt.ylabel('Price')
plt.show()

##### 1. Why did you pick the specific chart?

Boxplots compactly compare the spread and extreme observations across the four OHLC variables.

##### 2. What is/are the insight(s) found from the chart?

The OHLC variables have similar broad distributions but show substantial spread and potential extreme values. High and Low naturally form the upper and lower boundaries.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This helps detect unusual observations and understand scale differences. The presence of extremes is informative in financial data, so removal should require evidence of data-quality problems.

#### Chart - 9

In [ ]:
plt.figure(figsize=(10,5))
plt.scatter(df['Open'], df['Close'], alpha=0.7)
plt.title('Opening Price vs Closing Price')
plt.xlabel('Open'); plt.ylabel('Close')
plt.show()

##### 1. Why did you pick the specific chart?

A scatter plot tests the direct relationship between the opening and closing prices.

##### 2. What is/are the insight(s) found from the chart?

Open and Close show a strong positive relationship because both describe the same monthly price level, although observations do not fall exactly on one line.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The relationship suggests current-period price information can be useful for forecasting. It does not establish causality or guarantee future accuracy.

#### Chart - 10

In [ ]:
plt.figure(figsize=(10,5))
plt.scatter(df['Price_Range'], df['Monthly_Return_%'], alpha=0.7)
plt.title('Price Range vs Monthly Return')
plt.xlabel('High - Low'); plt.ylabel('Monthly Return (%)')
plt.show()

##### 1. Why did you pick the specific chart?

This chart checks whether larger trading ranges are associated with stronger or weaker monthly returns.

##### 2. What is/are the insight(s) found from the chart?

The relationship is dispersed, indicating that a wide price range does not by itself determine the direction of the monthly return.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The result cautions against relying on a single volatility measure. Combining range, lagged prices and returns is more informative for a forecasting model.

#### Chart - 11

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(df['Date'], df['Open_Close_Change_%'])
plt.axhline(0, linestyle='--')
plt.title('Monthly Open-to-Close Percentage Change')
plt.xlabel('Date'); plt.ylabel('Change (%)')
plt.show()

##### 1. Why did you pick the specific chart?

The open-to-close change focuses on the direction and magnitude of the within-month move.

##### 2. What is/are the insight(s) found from the chart?

There are both positive and negative monthly open-to-close changes, with some periods showing unusually large moves.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This measure can support historical risk analysis. Large negative changes are warning periods rather than evidence of positive growth.

#### Chart - 12

In [ ]:
yearly_close = df.groupby('Year')['Close'].mean()
plt.figure(figsize=(12,5))
plt.bar(yearly_close.index.astype(str), yearly_close.values)
plt.title('Average Closing Price by Year')
plt.xlabel('Year'); plt.ylabel('Average Close')
plt.xticks(rotation=45)
plt.show()

##### 1. Why did you pick the specific chart?

Year-level averages reduce monthly noise and make broad changes in the price regime easier to compare.

##### 2. What is/are the insight(s) found from the chart?

Average closing prices vary considerably across years, reflecting the changing level of the stock over the sample.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yearly aggregation can help communicate long-term historical movement, but it should not be used alone for forecasting or investment decisions.

#### Chart - 13

In [ ]:
df['Drawdown_%'] = (df['Close'] / df['Close'].cummax() - 1) * 100
plt.figure(figsize=(14,5))
plt.plot(df['Date'], df['Drawdown_%'])
plt.axhline(0, linestyle='--')
plt.title('Closing Price Drawdown from Previous Running High')
plt.xlabel('Date'); plt.ylabel('Drawdown (%)')
plt.show()

##### 1. Why did you pick the specific chart?

Drawdown directly measures how far the current closing price is below its previous running maximum, making downside episodes easy to compare.

##### 2. What is/are the insight(s) found from the chart?

The series contains periods of substantial drawdown from earlier highs, including a major decline near the end of the sample.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Drawdown is particularly relevant for risk management because it quantifies historical downside from prior peaks. Large drawdowns represent negative financial impact.

#### Chart - 14 - Correlation Heatmap

In [ ]:
numeric_cols = ['Open','High','Low','Close','Monthly_Return_%','Price_Range','Open_Close_Change_%']
plt.figure(figsize=(10,7))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

##### 1. Why did you pick the specific chart?

A correlation heatmap gives a compact view of linear relationships among the numerical variables and helps identify redundant features.

##### 2. What is/are the insight(s) found from the chart?

The raw OHLC price variables are strongly related because they represent different aspects of the same monthly price. Return and range variables capture different movement characteristics.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code
# Use pandas' scatter-matrix implementation for broad compatibility.
from pandas.plotting import scatter_matrix

pair_cols = ['Open','High','Low','Close']
axes = scatter_matrix(df[pair_cols], figsize=(12,12), diagonal='hist', alpha=0.7)
plt.suptitle('Pair Plot of OHLC Variables', y=1.02)
plt.show()

##### 1. Why did you pick the specific chart?

A pair plot visualizes pairwise relationships and distributions across the main OHLC variables.

##### 2. What is/are the insight(s) found from the chart?

The OHLC variables show strong positive pairwise relationships, while the distributions reveal their differing spreads and possible extreme values.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Missing-value check
print("Missing values before preprocessing:")
print(df.isnull().sum())

No missing-value imputation is required because the source dataset contains **0 missing values** in every column. The ML feature-engineering step can therefore proceed without introducing synthetic values.

Answer Here.

### 2. Handling Outliers

In [ ]:
# Inspect potential outliers using IQR, without deleting genuine market observations.
price_cols = ['Open','High','Low','Close']
Q1 = df[price_cols].quantile(0.25)
Q3 = df[price_cols].quantile(0.75)
IQR = Q3 - Q1
outlier_counts = ((df[price_cols] < (Q1 - 1.5*IQR)) | (df[price_cols] > (Q3 + 1.5*IQR))).sum()
print("Potential IQR outlier counts:")
print(outlier_counts)
print("\nNo rows are removed because financial extremes may be genuine market events.")

No aggressive outlier deletion was performed. Stock-price extremes can represent genuine market events rather than data errors. Removing them would risk deleting the very periods the model needs to learn. Distribution checks and robust tree-based models are therefore preferred over arbitrary row removal.

Answer Here.

### 3. Categorical Encoding

In [ ]:
# No categorical columns are present.
categorical_cols = df.select_dtypes(include=['object','category']).columns.tolist()
print("Categorical columns:", categorical_cols)
print("No categorical encoding required after Date conversion.")

No categorical variables are present in the dataset, so categorical encoding is not required. `Date` is converted to a datetime representation and useful temporal components are derived instead of one-hot encoding every date.

Answer Here.

### Textual Data Preprocessing
This section is **not applicable**. The dataset contains no natural-language/text field. `Date` is a structured temporal variable, not textual data for NLP processing.

#### 1. Expand Contraction

In [ ]:
# Not applicable: this dataset contains no textual/NLP column.

#### 2. Lower Casing

In [ ]:
# Not applicable: this dataset contains no textual/NLP column.

#### 3. Removing Punctuations

In [ ]:
# Not applicable: this dataset contains no textual/NLP column.

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Not applicable: this dataset contains no textual/NLP column.

#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Not applicable: this dataset contains no textual/NLP column.

In [ ]:
# Not applicable: this dataset contains no textual/NLP column.

#### 6. Rephrase Text

In [ ]:
# Not applicable: this dataset contains no textual/NLP column.

#### 7. Tokenization

In [ ]:
# Not applicable: this dataset contains no textual/NLP column.

#### 8. Text Normalization

In [ ]:
# Not applicable: this dataset contains no textual/NLP column.

Not applicable because the dataset contains no textual column. No stemming or lemmatization is appropriate for stock-price data.

Answer Here.

#### 9. Part of speech tagging

In [ ]:
# Not applicable: this dataset contains no textual/NLP column.

#### 10. Text Vectorization

In [ ]:
# Not applicable: this dataset contains no textual/NLP column.

Not applicable because there is no natural-language text. Numerical/time-series features are used instead of TF-IDF, CountVectorizer, or other text-vectorization methods.

Answer Here.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Build forecasting features from current and historical information only.
model_df = df[['Date','Open','High','Low','Close']].copy()
model_df['Return_1'] = model_df['Close'].pct_change()
model_df['Range'] = model_df['High'] - model_df['Low']
model_df['Open_Close_Pct'] = (model_df['Close'] - model_df['Open']) / model_df['Open']

for lag in [1, 2, 3]:
    model_df[f'Close_Lag_{lag}'] = model_df['Close'].shift(lag)

model_df['Close_Rolling_Mean_3'] = model_df['Close'].rolling(3).mean()
model_df['Close_Rolling_Mean_6'] = model_df['Close'].rolling(6).mean()
model_df['Return_Rolling_Std_3'] = model_df['Return_1'].rolling(3).std()

# Target is the next month's closing price.
model_df['Target_Next_Close'] = model_df['Close'].shift(-1)

print(model_df.head(10))

#### 2. Feature Selection

In [ ]:
feature_cols = [
    'Open','High','Low','Close','Return_1','Range','Open_Close_Pct',
    'Close_Lag_1','Close_Lag_2','Close_Lag_3',
    'Close_Rolling_Mean_3','Close_Rolling_Mean_6','Return_Rolling_Std_3'
]
target_col = 'Target_Next_Close'

ml_df = model_df.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)
X = ml_df[feature_cols]
y = ml_df[target_col]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeatures:", feature_cols)

Feature selection is performed by removing identifiers/non-predictive raw date fields and by using a deliberately small set of lag, rolling, return, and price-range features. Tree-based models also provide feature-importance estimates that help assess which engineered variables contribute to prediction.

Answer Here.

The most important candidate features are expected to be recent closing-price lags, rolling averages/volatility, current closing price, and recent return/range measures. Their actual importance is calculated in the model section rather than assumed beforehand.

Answer Here.

### 5. Data Transformation

Yes. Raw stock prices are non-stationary and can span a wide range over the 2005-2020 period. Percentage returns and relative price-range features are therefore created to describe movement and volatility. The raw price level is retained because the target itself is a price.

In [ ]:
# Relative/return transformations were created during feature engineering.
# Show their summary statistics.
X[['Return_1','Open_Close_Pct','Range']].describe().T

### 6. Data Scaling

In [ ]:
# Fit scaling only on the training set later to prevent leakage.
scaler = StandardScaler()
print("StandardScaler prepared for Linear Regression.")

StandardScaler is used for the Linear Regression model because scaling places engineered numerical features on comparable magnitudes. Tree-based models do not require scaling, so their original feature values are retained.

### 7. Dimesionality Reduction

A major dimensionality-reduction method such as PCA is not necessary here. The engineered feature set is intentionally small and interpretable, and reducing it would make the time-series relationships harder to explain.

No PCA or other dimensionality-reduction algorithm is applied. The final feature set contains a manageable number of domain-driven variables, so interpretability is more valuable than compressing the feature space.

In [ ]:
# PCA is intentionally not applied because the engineered feature set is small and interpretable.
print("Dimensionality reduction skipped: feature count is small and domain-driven.")

Not applicable because dimensionality reduction was intentionally not performed. The feature count is already small after feature engineering.

Answer Here.

### 8. Data Splitting

In [ ]:
# Chronological split: no shuffling.
split_idx = int(len(ml_df) * 0.80)
X_train = X.iloc[:split_idx].copy()
X_test = X.iloc[split_idx:].copy()
y_train = y.iloc[:split_idx].copy()
y_test = y.iloc[split_idx:].copy()

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training period:", ml_df.loc[:split_idx-1, 'Date'].min().date(), "to", ml_df.loc[:split_idx-1, 'Date'].max().date())
print("Testing period:", ml_df.loc[split_idx:, 'Date'].min().date(), "to", ml_df.loc[split_idx:, 'Date'].max().date())

An **80:20 chronological split** is used. The first 80% of observations are used for training and the final 20% are reserved as an unseen future test period. Random splitting is avoided because it can leak future market regimes into training.

Answer Here.

### 9. Handling Imbalanced Dataset

The target is continuous rather than a class label, so the concept of class imbalance does not apply. This is a regression problem. The target distribution is examined, but no SMOTE/undersampling/class-weight method is appropriate.

Answer Here.

In [ ]:
# Regression target has no class imbalance.
print("Target is continuous. No class-imbalance treatment is required.")

No imbalance treatment is performed because this is regression, not classification. Applying class-balancing techniques would be inappropriate.

Answer Here.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE_%': mape}

# Linear Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
pred_lr = lr.predict(X_test_scaled)

lr_metrics = regression_metrics(y_test, pred_lr)
lr_metrics

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
lr_metric_series = pd.Series(lr_metrics)
plt.figure(figsize=(8,4))
lr_metric_series.plot(kind='bar')
plt.title('Linear Regression - Test Metrics')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Tune Linear Regression using TimeSeriesSplit.
tscv = TimeSeriesSplit(n_splits=5)
lr_grid = {'fit_intercept': [True, False]}

lr_search = GridSearchCV(
    LinearRegression(),
    lr_grid,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
lr_search.fit(X_train_scaled, y_train)

lr_tuned = lr_search.best_estimator_
pred_lr_tuned = lr_tuned.predict(X_test_scaled)
lr_tuned_metrics = regression_metrics(y_test, pred_lr_tuned)

print("Best parameters:", lr_search.best_params_)
print(lr_tuned_metrics)

GridSearchCV with TimeSeriesSplit is used because the dataset is chronological and relatively small. TimeSeriesSplit preserves temporal order while evaluating multiple hyperparameter combinations without random future-to-past mixing.

GridSearchCV is selected because it gives a transparent, reproducible search over a small hyperparameter grid. TimeSeriesSplit is used as the cross-validation strategy so validation data always occurs after the corresponding training data.

The notebook calculates the tuned model's test metrics and compares them with the untuned baseline. Improvement is judged using lower MAE/RMSE and higher R² on the same chronological test set.

The tuned Linear Regression results are calculated above. Because only `fit_intercept` is searched, improvement may be small. The same held-out test set is used for the baseline and tuned versions, so the comparison remains consistent.

### ML Model - 2

**Random Forest Regressor** is used as a nonlinear ensemble baseline.

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
rf_baseline = RandomForestRegressor(n_estimators=300, random_state=42, min_samples_leaf=2)
rf_baseline.fit(X_train, y_train)
pred_rf = rf_baseline.predict(X_test)
rf_metrics = regression_metrics(y_test, pred_rf)

pd.Series(rf_metrics).plot(kind='bar', figsize=(8,4))
plt.title('Random Forest - Test Metrics')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.show()

print(rf_metrics)

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
rf_grid_params = {
    'n_estimators': [100, 200],
    'max_depth': [None, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 1.0]
}

rf_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    rf_grid_params,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
rf_search.fit(X_train, y_train)

rf_tuned = rf_search.best_estimator_
pred_rf_tuned = rf_tuned.predict(X_test)
rf_tuned_metrics = regression_metrics(y_test, pred_rf_tuned)

print("Best parameters:", rf_search.best_params_)
print(rf_tuned_metrics)

GridSearchCV with TimeSeriesSplit is used again for the second model. This keeps the evaluation procedure consistent across models and avoids an optimistic random cross-validation estimate.

A small, interpretable grid is appropriate because only 185 original observations are available. Excessively large hyperparameter searches would increase variance and computation without guaranteeing better generalisation.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The tuned Random Forest is compared with the untuned Random Forest using the same test period. Lower RMSE/MAE and higher R² indicate improvement. The actual values are produced in the next model-comparison cell.

- **MAE** shows the average absolute pricing error and is easy to interpret in the stock-price units.
- **RMSE** penalizes larger mistakes more heavily, which is useful when large forecasting errors are costly.
- **R²** measures how much of the target variation is explained by the model, but it should not be interpreted as a guarantee of profitable trading.
- **MAPE** expresses error as a percentage, but it can become unstable when actual prices are close to zero.

For a business/decision-support use case, lower MAE and RMSE are desirable because they reduce the size of forecasting errors. Model outputs should be treated as analytical forecasts rather than investment guarantees.

Answer Here.

### ML Model - 3

In [ ]:
# Gradient Boosting Regressor
gbr = GradientBoostingRegressor(random_state=42)
gbr.fit(X_train, y_train)
pred_gbr = gbr.predict(X_test)
gbr_metrics = regression_metrics(y_test, pred_gbr)

print(gbr_metrics)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
pd.Series(gbr_metrics).plot(kind='bar', figsize=(8,4))
plt.title('Gradient Boosting - Test Metrics')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
gbr_grid_params = {
    'n_estimators': [50, 100],
    'learning_rate': [0.05, 0.1],
    'max_depth': [1, 2, 3],
    'min_samples_leaf': [2, 4]
}

gbr_search = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    gbr_grid_params,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
gbr_search.fit(X_train, y_train)

gbr_tuned = gbr_search.best_estimator_
pred_gbr_tuned = gbr_tuned.predict(X_test)
gbr_tuned_metrics = regression_metrics(y_test, pred_gbr_tuned)

print("Best parameters:", gbr_search.best_params_)
print(gbr_tuned_metrics)

# Consolidated comparison
model_comparison = pd.DataFrame({
    'Linear Regression': lr_tuned_metrics,
    'Random Forest': rf_tuned_metrics,
    'Gradient Boosting': gbr_tuned_metrics
}).T.sort_values('RMSE')

print("\nFinal model comparison:")
display(model_comparison)

best_model_name = model_comparison.index[0]
print("Selected final model:", best_model_name)
# Plot actual vs predicted values for the selected model
predictions = {
    'Linear Regression': pred_lr_tuned,
    'Random Forest': pred_rf_tuned,
    'Gradient Boosting': pred_gbr_tuned
}
best_pred = predictions[best_model_name]

plt.figure(figsize=(12,5))
plt.plot(ml_df.loc[split_idx:, 'Date'].values, y_test.values, marker='o', label='Actual')
plt.plot(ml_df.loc[split_idx:, 'Date'].values, best_pred, marker='o', label='Predicted')
plt.title(f'Actual vs Predicted Next-Month Close - {best_model_name}')
plt.xlabel('Test Period')
plt.ylabel('Next-Month Close')
plt.legend()
plt.show()

# Feature importance when the selected model supports it
if best_model_name == 'Random Forest':
    importance = pd.Series(rf_tuned.feature_importances_, index=feature_cols).sort_values(ascending=False)
elif best_model_name == 'Gradient Boosting':
    importance = pd.Series(gbr_tuned.feature_importances_, index=feature_cols).sort_values(ascending=False)
else:
    importance = pd.Series(np.abs(lr_tuned.coef_), index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10,6))
importance.head(10).sort_values().plot(kind='barh')
plt.title(f'Top Feature Contributions - {best_model_name}')
plt.xlabel('Importance / Absolute Coefficient')
plt.show()

print("Top features:")
print(importance.head(10))


##### Which hyperparameter optimization technique have you used and why?

The tuned Gradient Boosting model is evaluated on the same unseen chronological test period. Hyperparameter tuning is considered useful only if the held-out metrics improve rather than merely the cross-validation score.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The consolidated comparison is generated in the previous cell. The model with the lowest chronological test RMSE is selected as the final candidate, with MAE, R² and MAPE retained for context.

The final model is selected using the chronological test-set comparison of MAE, RMSE, R² and MAPE. The notebook automatically identifies the best candidate using **lowest RMSE**, while the other metrics are reported alongside it. This is preferable to selecting a model only because it fits the training data well.

The final prediction model is selected automatically from the tuned Linear Regression, Random Forest, and Gradient Boosting models using the **lowest test RMSE**. The comparison table generated above also reports MAE, R² and MAPE so the selection is not based on a single hidden criterion.

# **Conclusion**

## Conclusion
This project analysed 185 monthly Yes Bank stock-price observations and developed a supervised regression workflow for predicting the next month's closing price. The workflow covers data validation, exploratory analysis, time-series feature engineering, chronological train/test splitting, model training, time-series cross-validation, hyperparameter tuning, and multi-metric evaluation.

The main methodological constraint is the limited number of observations and the non-stationary nature of financial prices. Therefore, the results should be interpreted as an educational forecasting exercise rather than evidence of a reliable trading strategy. The final model is selected from the tested candidates using held-out chronological performance, with RMSE as the primary selection metric.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***